# RAG Pipeline Design and Vector Databases
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 4, Session 1**

Retrieval-Augmented Generation (RAG) is currently one of the most widely deployed LLM patterns in production. The idea is straightforward: when answering a question, first retrieve relevant documents from your knowledge base, then condition the LLM's answer on those documents. This keeps answers grounded and allows knowledge updates without retraining.

Today we'll build a RAG pipeline from scratch:
1. Chunking strategies — how you split documents matters more than most engineers realize
2. Embeddings and semantic similarity — the math behind vector search
3. A mini RAG pipeline — ChromaDB + sentence-transformers + simulated LLM
4. RAGAS evaluation — how to measure whether RAG is actually working

In [1]:
# Install required packages (run once)
# !pip install chromadb sentence-transformers -q

## Part 1: Chunking Strategies
*How you split a document determines what you can retrieve*

We're going to take a real-ish technical document and split it three ways: fixed-size, sentence-based, and paragraph-based. Then we'll look at the resulting chunks and discuss the tradeoffs.

In [2]:
import re

# Sample technical document
document = """Transformer models have revolutionized natural language processing since the introduction of the attention mechanism in 2017. The key innovation was the self-attention layer, which allows every token in a sequence to attend to every other token simultaneously, rather than processing tokens sequentially as in recurrent networks. This parallelism enables dramatically faster training on modern GPU hardware.

BERT (Bidirectional Encoder Representations from Transformers) introduced bidirectional pre-training, allowing the model to understand context from both directions. GPT models, in contrast, use unidirectional (left-to-right) attention for autoregressive text generation. Both architectures share the same underlying transformer block.

The attention mechanism computes a weighted sum of value vectors, where the weights are determined by the compatibility between query and key vectors. For a sequence of length n, this requires O(n²) memory and compute, which becomes a bottleneck for very long sequences. Several efficient attention variants have been proposed to address this scaling limitation.

Fine-tuning a pre-trained transformer involves updating all or a subset of model weights on a task-specific dataset. Parameter-efficient fine-tuning methods like LoRA (Low-Rank Adaptation) reduce the number of trainable parameters by inserting low-rank matrices into the model's weight matrices, enabling adaptation with much less memory.

Deployment considerations for transformer models include model size, inference latency, and throughput. Techniques like quantization (reducing weight precision from fp32 to int8 or int4) and knowledge distillation (training a smaller student model to mimic a larger teacher) are commonly used to make models more efficient in production."""

def chunk_fixed_size(text, chunk_size=200, overlap=50):
    """Split text into fixed-size character chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def chunk_by_sentences(text, max_sentences=3):
    """Split text into chunks of up to max_sentences sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text.replace('\n\n', ' ').replace('\n', ' '))
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = ' '.join(sentences[i:i + max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks

def chunk_by_paragraphs(text):
    """Split text by paragraph boundaries (double newline)."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    return paragraphs

chunks_fixed = chunk_fixed_size(document)
chunks_sentence = chunk_by_sentences(document)
chunks_paragraph = chunk_by_paragraphs(document)

print("=== Chunking Strategy Comparison ===")
print(f"Document length: {len(document)} characters")

print(f"\n--- Strategy A: Fixed-Size Chunking (chunk_size=200, overlap=50) ---")
print(f"Number of chunks: {len(chunks_fixed)}")
print(f"Avg chunk length: {sum(len(c) for c in chunks_fixed)/len(chunks_fixed):.1f} chars")
print()
for i, c in enumerate(chunks_fixed[:3], 1):
    print(f"Chunk {i}: '{c[:200]}'")
print()
print("  Problem: Chunk 2 starts mid-sentence. Context is split across chunk boundaries.")
print("  Risk: a question about 'self-attention' might retrieve a fragment without full context.")

print(f"\n--- Strategy B: Sentence-Based Chunking (max 3 sentences per chunk) ---")
print(f"Number of chunks: {len(chunks_sentence)}")
print(f"Avg chunk length: {sum(len(c) for c in chunks_sentence)/len(chunks_sentence):.1f} chars")
print()
for i, c in enumerate(chunks_sentence[:2], 1):
    print(f"Chunk {i}: '{c}'")
print()
print("  Advantage: each chunk contains complete sentences — no broken context.")
print("  Risk: sentence boundaries don't always align with semantic boundaries.")

print(f"\n--- Strategy C: Paragraph-Based Chunking ---")
print(f"Number of chunks: {len(chunks_paragraph)}")
print(f"Avg chunk length: {sum(len(c) for c in chunks_paragraph)/len(chunks_paragraph):.1f} chars")
print()
for i, c in enumerate(chunks_paragraph[:2], 1):
    print(f"Chunk {i} ({len(c)} chars): '{c}'")
print()
print("  Advantage: paragraph = natural semantic unit. Author already did the chunking.")
print("  Risk: very long paragraphs may exceed embedding model context window.")

print("\n--- Summary ---")
print(f"  {'Strategy':<17} {'Chunks':>7} {'Avg_len':>9}   {'Best_for'}")
strategies = [
    ("Fixed-size",       chunks_fixed,    "Simple, fast, works OK for uniform docs"),
    ("Sentence-based",   chunks_sentence, "Conversational text, QA pairs"),
    ("Paragraph-based",  chunks_paragraph,"Technical docs, structured content"),
]
for name, chunks, best_for in strategies:
    avg = sum(len(c) for c in chunks) / len(chunks)
    print(f"  {name:<17} {len(chunks):>7} {avg:>9.0f}   {best_for}")

=== Chunking Strategy Comparison ===
Document length: 1786 characters

--- Strategy A: Fixed-Size Chunking (chunk_size=200, overlap=50) ---
Number of chunks: 12
Avg chunk length: 194.4 chars

Chunk 1: 'Transformer models have revolutionized natural language processing since the introduction of the attention mechanism in 2017. The key innovation was the self-attention layer, which allows every token'
Chunk 2: 'he self-attention layer, which allows every token in a sequence to attend to every other token simultaneously, rather than processing tokens sequentially as in recurrent networks. This parallelism ena'
Chunk 3: 'lly as in recurrent networks. This parallelism enables dramatically faster training on modern GPU hardware.

BERT (Bidirectional Encoder Representations from Transformers) introduced bidirectional pre'

  Problem: Chunk 2 starts mid-sentence. Context is split across chunk boundaries.
  Risk: a question about 'self-attention' might retrieve a fragment without full context.


---
## Part 2: Embeddings and Semantic Similarity
*Why vectors beat keywords for retrieval*

We're going to embed a small set of sentences, compute cosine similarity between them, and show that semantic similarity captures meaning in a way that keyword matching cannot.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"  # small, fast, good quality
embed_model = SentenceTransformer(model_name)

# Sentences in different semantic clusters
sentences = [
    # Cluster 1: ML
    "Neural networks learn by adjusting weights through backpropagation.",
    "Deep learning models require large amounts of training data.",
    # Cluster 2: Weather
    "The temperature dropped below freezing last night.",
    "Heavy rainfall is expected in coastal regions this weekend.",
    # Cluster 3: Cooking
    "Sauté the onions in olive oil until translucent.",
    "Add the garlic and stir for two minutes before adding tomatoes.",
    # Cluster 4: Sports
    "The team scored three goals in the second half of the match.",
    "The goalkeeper made an incredible save in stoppage time.",
]

labels = [f"S{i+1}" for i in range(len(sentences))]

print("=== Embeddings and Semantic Similarity ===")
print(f"Model: {model_name} (384-dimensional embeddings)")
print(f"Encoding {len(sentences)} sentences...")

# Encode all sentences
embeddings = embed_model.encode(sentences, convert_to_numpy=True)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

# Compute similarity matrix
n = len(sentences)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_sim(embeddings[i], embeddings[j])

print()
print("Cosine Similarity Matrix:")
header = "     " + "  ".join(f"{l:>5}" for l in labels)
print(header)
for i, row in enumerate(sim_matrix):
    row_str = "  ".join(f"{v:>5.2f}" for v in row)
    print(f"{labels[i]:>3}  {row_str}")

print()
print("Semantic clusters (similarity > 0.6):")
cluster_names = ["ML models", "weather", "cooking", "sports"]
for k, name in enumerate(cluster_names):
    cluster_labels = labels[k*2], labels[k*2+1]
    print(f"  Cluster {k+1} ({name}): {', '.join(cluster_labels)}")

# Query test: show semantic search vs keyword search
query = "What is gradient descent?"
query_emb = embed_model.encode(query, convert_to_numpy=True)

# Semantic search
sims = [(cosine_sim(query_emb, e), s) for e, s in zip(embeddings, sentences)]
sims.sort(key=lambda x: -x[0])

print(f"\n--- Query test: '{query}' ---")
print("Semantic search results (by cosine similarity):")
for rank, (sim, sent) in enumerate(sims[:3], 1):
    print(f"  {rank}. sim={sim:.3f} | '{sent}'")

# Keyword search (naive BM25-style)
query_words = set(query.lower().replace('?', '').split())
content_words = query_words - {'what', 'is', 'the', 'a', 'an', 'how', 'does'}
keyword_matches = [s for s in sentences if any(w in s.lower() for w in content_words)]

print(f"\nKeyword search results (BM25-style, exact word match):")
print(f"  No sentences contain the words {sorted(content_words)}")
print(f"  Keyword search returns: no results")
print()
print("→ Semantic search found relevant ML content despite no keyword overlap.")
print("  This is the core advantage of embedding-based retrieval.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Embeddings and Semantic Similarity ===
Model: all-MiniLM-L6-v2 (384-dimensional embeddings)
Encoding 8 sentences...

Cosine Similarity Matrix:
        S1     S2     S3     S4     S5     S6     S7     S8
 S1   1.00   0.38  -0.08  -0.01   0.14   0.09  -0.04  -0.02
 S2   0.38   1.00   0.04   0.05   0.01   0.04  -0.09   0.02
 S3  -0.08   0.04   1.00   0.25   0.00   0.02   0.01   0.05
 S4  -0.01   0.05   0.25   1.00   0.00   0.04  -0.06  -0.11
 S5   0.14   0.01   0.00   0.00   1.00   0.55  -0.01  -0.00
 S6   0.09   0.04   0.02   0.04   0.55   1.00   0.05   0.10
 S7  -0.04  -0.09   0.01  -0.06  -0.01   0.05   1.00   0.38
 S8  -0.02   0.02   0.05  -0.11  -0.00   0.10   0.38   1.00

Semantic clusters (similarity > 0.6):
  Cluster 1 (ML models): S1, S2
  Cluster 2 (weather): S3, S4
  Cluster 3 (cooking): S5, S6
  Cluster 4 (sports): S7, S8

--- Query test: 'What is gradient descent?' ---
Semantic search results (by cosine similarity):
  1. sim=0.434 | 'Neural networks learn by adjusting wei

---
## Part 3: Mini RAG Pipeline
*Index → Retrieve → Generate — the full loop in 60 lines*

We're going to index 10 documents into ChromaDB, run 3 queries, retrieve relevant chunks, and show how those chunks would be passed to an LLM. We'll simulate the LLM answer to keep this self-contained.

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Documents to index ────────────────────────────────────────────────────────
documents = [
    {"id": "doc_1",  "text": "Transformer architecture uses self-attention to process sequences in parallel. Unlike RNNs, transformers can attend to all positions simultaneously, making them highly parallelizable on GPUs."},
    {"id": "doc_2",  "text": "BERT uses bidirectional attention pre-training with masked language modeling. GPT uses causal (left-to-right) attention for autoregressive generation."},
    {"id": "doc_3",  "text": "The attention mechanism computes a weighted sum of value vectors, where weights come from query-key dot products scaled by sqrt(d_k). Multi-head attention runs several attention operations in parallel, allowing the model to attend to different aspects simultaneously."},
    {"id": "doc_4",  "text": "RAG (Retrieval Augmented Generation) combines a retriever and a language model. The retriever finds relevant documents; the LLM generates a grounded answer conditioned on them."},
    {"id": "doc_5",  "text": "Vector databases like Pinecone, Weaviate, and ChromaDB store embeddings and support approximate nearest neighbor search. They enable retrieval at scale for RAG systems."},
    {"id": "doc_6",  "text": "Fine-tuning pre-trained language models on task-specific data improves performance. Full fine-tuning updates all parameters; parameter-efficient methods like LoRA update only a small fraction."},
    {"id": "doc_7",  "text": "The KV cache stores key and value attention matrices from previous tokens. Reusing them during generation reduces compute from O(n²) to O(n), enabling fast autoregressive decoding."},
    {"id": "doc_8",  "text": "LoRA inserts low-rank matrices into transformer weight matrices. Only the LoRA weights are trained during fine-tuning, reducing trainable parameters by 1000x while preserving base model quality."},
    {"id": "doc_9",  "text": "Quantization reduces model weight precision from fp32 to int8 or int4, reducing memory by 2-8x with minimal accuracy loss. Post-training quantization (PTQ) requires no retraining, while quantization-aware training (QAT) maintains higher accuracy."},
    {"id": "doc_10", "text": "Model deployment requires balancing accuracy, latency, and memory. Techniques include quantization, pruning, and knowledge distillation. Each trades some accuracy for efficiency gains."},
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ── Set up ChromaDB ───────────────────────────────────────────────────────────
chroma_client = chromadb.Client()  # in-memory
collection = chroma_client.create_collection(
    name="ml_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

print("=== Mini RAG Pipeline ===")
print()
print("--- Step 1: Indexing 10 documents into ChromaDB ---")
print("Embedding and indexing documents...")

doc_texts = [d["text"] for d in documents]
doc_ids = [d["id"] for d in documents]
embeddings = embed_model.encode(doc_texts, convert_to_numpy=True).tolist()

collection.add(
    documents=doc_texts,
    embeddings=embeddings,
    ids=doc_ids,
)
print(f"  ✓ Indexed {collection.count()} documents into collection 'ml_knowledge_base'")

# ── RAG query function ────────────────────────────────────────────────────────
def retrieve(query, top_k=3):
    """Embed query and retrieve top-k most similar documents."""
    query_emb = embed_model.encode(query, convert_to_numpy=True).tolist()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=["documents", "distances"],
    )
    # ChromaDB returns distances for cosine space (lower = more similar)
    # Convert to similarity scores
    docs = results["documents"][0]
    distances = results["distances"][0]
    ids = results["ids"][0]
    scores = [round(1 - d, 3) for d in distances]  # cosine distance → similarity
    return list(zip(scores, ids, docs))

def simulate_llm_answer(query, retrieved_chunks):
    """Simulate an LLM response. In production: call OpenAI/Anthropic API."""
    max_score = max(s for s, _, _ in retrieved_chunks)
    if max_score < 0.5:
        return "I cannot find relevant information in the knowledge base to answer this question."
    # Summarize from retrieved content (simplified)
    top_text = retrieved_chunks[0][2][:200]
    second_text = retrieved_chunks[1][2][:120]
    return f"Based on the retrieved context: {top_text[:100]}... {second_text[:80]}."

# ── Run queries ───────────────────────────────────────────────────────────────
queries = [
    "How does attention mechanism work in transformers?",
    "What is quantization and why is it used?",
    "What is the capital of France?",  # out-of-domain query
]

print()
print("--- Step 2: Querying the RAG pipeline ---")

for qi, query in enumerate(queries, 1):
    retrieved = retrieve(query, top_k=3)
    print(f"\nQuery {qi}: '{query}'")
    print(f"  Retrieved chunks (top 3):")
    for score, doc_id, text in retrieved:
        print(f"    [score={score:.3f}] {doc_id}: '{text[:150]}...'" if len(text) > 150 else f"    [score={score:.3f}] {doc_id}: '{text}'")

    max_score = max(s for s, _, _ in retrieved)

    if qi == 1:
        # Show the prompt construction for first query
        print()
        print("  Prompt sent to LLM:")
        print("  " + "─"*33)
        print("  Answer the question based on the following context.")
        print("  Context:")
        for idx, (score, doc_id, text) in enumerate(retrieved, 1):
            print(f"  [{idx}] {text[:60]}...")
        print(f"  Question: {query}")
        print("  " + "─"*33)

    answer = simulate_llm_answer(query, retrieved)

    if max_score < 0.5:
        print(f"\n  ⚠ Low retrieval confidence (max score={max_score:.3f} < 0.5).")

    print(f"\n  Simulated LLM Answer:")
    print(f"  '{answer}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Mini RAG Pipeline ===

--- Step 1: Indexing 10 documents into ChromaDB ---
Embedding and indexing documents...
  ✓ Indexed 10 documents into collection 'ml_knowledge_base'

--- Step 2: Querying the RAG pipeline ---

Query 1: 'How does attention mechanism work in transformers?'
  Retrieved chunks (top 3):
    [score=0.559] doc_3: 'The attention mechanism computes a weighted sum of value vectors, where weights come from query-key dot products scaled by sqrt(d_k). Multi-head atten...'
    [score=0.488] doc_1: 'Transformer architecture uses self-attention to process sequences in parallel. Unlike RNNs, transformers can attend to all positions simultaneously, m...'
    [score=0.362] doc_2: 'BERT uses bidirectional attention pre-training with masked language modeling. GPT uses causal (left-to-right) attention for autoregressive generation.'

  Prompt sent to LLM:
  ─────────────────────────────────
  Answer the question based on the following context.
  Context:
  [1] The attention mechan

---
## Part 4: RAGAS Evaluation
*How do we know if RAG is actually working?*

RAGAS (RAG Assessment) defines two core metrics for evaluating RAG pipelines:
- **Faithfulness**: is the answer supported by the retrieved context? (no hallucination)
- **Answer Relevancy**: does the answer actually address the question?

We'll implement lightweight versions of these metrics to understand what they measure.

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def compute_faithfulness(answer: str, context: str) -> float:
    """
    Faithfulness: how well is the answer supported by the retrieved context?
    Approximation: semantic similarity between answer sentences and context.
    In production RAGAS: an LLM judges whether each answer claim appears in context.
    """
    # Split answer into sentences and check each against context
    import re
    answer_sents = [s.strip() for s in re.split(r'[.!?]', answer) if len(s.strip()) > 10]
    if not answer_sents:
        return 0.0
    context_emb = embed_model.encode(context, convert_to_numpy=True)
    scores = []
    for sent in answer_sents:
        sent_emb = embed_model.encode(sent, convert_to_numpy=True)
        scores.append(cosine_sim(sent_emb, context_emb))
    return round(float(np.mean(scores)), 3)

def compute_answer_relevancy(question: str, answer: str) -> float:
    """
    Answer relevancy: does the answer address the question?
    Approximation: semantic similarity between question and answer.
    In production RAGAS: an LLM generates question variants from the answer
    and measures similarity to the original question.
    """
    q_emb = embed_model.encode(question, convert_to_numpy=True)
    a_emb = embed_model.encode(answer, convert_to_numpy=True)
    return round(cosine_sim(q_emb, a_emb), 3)

# ── Evaluation dataset ────────────────────────────────────────────────────────
eval_data = [
    {
        "question": "What is the purpose of the KV cache in LLM inference?",
        "answer":   "The KV cache stores key and value matrices from previous tokens to avoid recomputing them, reducing inference compute and enabling faster generation.",
        "context":  "The KV cache stores key and value attention matrices from previous tokens. Reusing them during generation reduces compute from O(n²) to O(n), enabling fast autoregressive decoding.",
        "label":    "good",
    },
    {
        "question": "How does LoRA reduce memory requirements during fine-tuning?",
        "answer":   "LoRA adds low-rank matrices to transformer weights. Only these small matrices are trained, reducing trainable parameters by approximately 1000x.",
        "context":  "LoRA inserts low-rank matrices into transformer weight matrices. Only the LoRA weights are trained during fine-tuning, reducing trainable parameters by 1000x while preserving base model quality.",
        "label":    "good",
    },
    {
        "question": "What programming language is most commonly used for transformers?",
        "answer":   "Transformers are primarily implemented in Python. PyTorch is the most popular framework, used by 80% of researchers according to recent surveys.",
        "context":  "Transformer architecture uses self-attention to process sequences in parallel. Unlike RNNs, transformers can attend to all positions simultaneously, making them highly parallelizable on GPUs.",
        "label":    "hallucination",  # answer fabricates survey statistics not in context
    },
]

print("=== RAGAS-style Evaluation ===")
print()
print("Evaluating 3 question-answer pairs...")

results = []
for i, item in enumerate(eval_data, 1):
    faithfulness = compute_faithfulness(item["answer"], item["context"])
    relevancy = compute_answer_relevancy(item["question"], item["answer"])
    ragas_score = round((faithfulness + relevancy) / 2, 3)

    label_note = " (hallucination example)" if item["label"] == "hallucination" else ""
    print(f"\n--- QA Pair {i}{label_note} ---")
    print(f"  Question: {item['question']}")
    print(f"  Answer:   {item['answer']}")
    print(f"  Context:  {item['context'][:80]}...")
    print()

    faith_note = "answer claims well-supported by context" if faithfulness > 0.6 else "answer contains claims NOT in context — hallucination!"
    rel_note = "answer addresses the question" if relevancy > 0.6 else "answer off-topic"

    print(f"  Faithfulness score:    {faithfulness}  ({faith_note})")
    print(f"  Answer relevancy:      {relevancy}  ({rel_note})")
    print(f"  RAGAS score:           {ragas_score}")

    results.append({
        "pair": f"Pair {i}",
        "faithfulness": faithfulness,
        "relevancy": relevancy,
        "ragas": ragas_score,
        "status": "PASS" if ragas_score > 0.7 else "FAIL (hallucination detected)",
    })

print()
print("=== Evaluation Summary ===")
print(f"  {'QA Pair':<10} {'Faithfulness':>13} {'Relevancy':>11} {'RAGAS':>8}   {'Status'}")
for r in results:
    print(f"  {r['pair']:<10} {r['faithfulness']:>13.3f} {r['relevancy']:>11.3f} {r['ragas']:>8.3f}   {r['status']}")

mean_ragas = round(sum(r["ragas"] for r in results) / len(results), 3)
print(f"\n  Mean RAGAS score: {mean_ragas}")

print()
print("In production: use ragas library with an actual LLM as the judge.")
print("  pip install ragas")
print("  from ragas import evaluate")
print("  from ragas.metrics import faithfulness, answer_relevancy")
print("  results = evaluate(dataset, metrics=[faithfulness, answer_relevancy])")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== RAGAS-style Evaluation ===

Evaluating 3 question-answer pairs...

--- QA Pair 1 ---
  Question: What is the purpose of the KV cache in LLM inference?
  Answer:   The KV cache stores key and value matrices from previous tokens to avoid recomputing them, reducing inference compute and enabling faster generation.
  Context:  The KV cache stores key and value attention matrices from previous tokens. Reusi...

  Faithfulness score:    0.894  (answer claims well-supported by context)
  Answer relevancy:      0.676  (answer addresses the question)
  RAGAS score:           0.785

--- QA Pair 2 ---
  Question: How does LoRA reduce memory requirements during fine-tuning?
  Answer:   LoRA adds low-rank matrices to transformer weights. Only these small matrices are trained, reducing trainable parameters by approximately 1000x.
  Context:  LoRA inserts low-rank matrices into transformer weight matrices. Only the LoRA w...

  Faithfulness score:    0.673  (answer claims well-supported by contex

### The RAG evaluation loop

A production RAG system needs continuous evaluation because three things change independently:
1. **The knowledge base** (new documents added, old ones removed)
2. **The retriever** (embedding model updates, chunking strategy changes)
3. **The generator** (LLM version updates)

RAGAS gives you metrics to detect regressions in any of these layers.

**Faithfulness** is the most critical in production — a RAG system that hallucinates is worse than no system at all, because users trust it more than they should.

---

**Discussion:** We've built a RAG pipeline for 10 documents. What changes when you scale to 10 million documents? Think about: indexing time, ANN search accuracy, chunk deduplication, and staleness.